<a href="https://colab.research.google.com/github/Jorge-Ruiz-Troccoli/Data-Science-III/blob/main/Introducci%C3%B3n_Transformers/NLP_reconocimiento_nombres_genero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🕵️‍♂️ Reconocimiento de Nombres y Apellidos con NLP

En este notebook vamos a construir dos clasificadores de texto usando técnicas de **NLP (Procesamiento de Lenguaje Natural)**:

1. 🧾 **Nombre vs. Apellido**: dado un string, predecir si corresponde a un nombre o a un apellido.
2. 🚻 **Género**: dado un nombre, predecir si probablemente pertenece a un hombre o una mujer.

El modelo principal que vamos a usar es **Naive Bayes**, un clásico para clasificación de texto. Más abajo vamos a ver en detalle en qué consiste y por qué es una buena elección para este problema. 👇


In [ ]:
import random
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score


## 🎲 ¿Qué es Naive Bayes?

**Naive Bayes** es una familia de clasificadores probabilísticos basados en el **teorema de Bayes**:

$$P(clase \mid texto) = \frac{P(texto \mid clase) \cdot P(clase)}{P(texto)}$$

En criollo: para cada clase posible (por ejemplo `nombre` u `apellido`), el modelo calcula *qué tan probable es esa clase, dado el texto que vimos*, combinando:

- 📊 **P(clase)**: qué tan frecuente es esa clase en general (el "prior").
- 🔤 **P(texto | clase)**: qué tan típicas son las palabras/combinaciones de letras del texto dentro de esa clase.

Se llama **"naive" (ingenuo)** porque asume que cada feature (cada palabra o n-grama) aporta información de forma **independiente** de las demás, lo cual casi nunca es 100% cierto en el lenguaje real... pero en la práctica funciona sorprendentemente bien. 🤷‍♂️

### 🧬 La variante que usamos: `MultinomialNB`

`MultinomialNB` es la versión de Naive Bayes pensada para **datos de conteo** (cuántas veces aparece cada palabra/n-grama en un texto), que es exactamente lo que producen `CountVectorizer` y `TfidfVectorizer`. Por eso es la combinación clásica para clasificación de texto.

### ✅ ¿Por qué es una buena elección para este caso?

- ⚡ **Rápido y liviano**: entrena en segundos incluso con miles de nombres/apellidos, algo clave cuando estamos iterando y comparando modelos.
- 🧩 **Funciona bien con datos dispersos (sparse) y de alta dimensión**: al vectorizar texto terminamos con miles de columnas (una por cada palabra/n-grama), y Naive Bayes maneja esto de forma muy eficiente.
- 📉 **Necesita poca data para generalizar decentemente**: al basarse en frecuencias simples, no requiere tantos ejemplos como modelos más complejos (redes neuronales, por ejemplo).
- 🏷️ **Ideal como baseline**: es el punto de partida estándar en clasificación de texto antes de probar modelos más sofisticados (SGD, SVM, transformers, etc.), y como vamos a ver, en este caso le gana o empata a un modelo lineal más "moderno" como `SGDClassifier`.
- 🔡 Nuestro problema es justamente eso: distinguir patrones de letras/combinaciones típicas de nombres vs. apellidos (o de nombres de hombre vs. mujer), algo que se basa mucho en la frecuencia de ciertas terminaciones y combinaciones de letras (ej: los nombres de mujer en español suelen terminar en "a"). Naive Bayes captura ese tipo de señal estadística muy bien.

📚 Si querés profundizar, podés leer más sobre Naive Bayes en Python acá: https://anderfernandez.com/blog/naive-bayes-en-python/


## 📂 Carga y preparación de los datos

In [ ]:
# Dataset de personas (nombre, apellido, sexo) tomado de:
# https://datos.gob.ar/dataset/mincyt-personal-ciencia-tecnologia
df = pd.read_csv("personas.csv", sep=",")
df = df.dropna()
df.head()


Para el primer modelo (nombre vs. apellido), necesitamos armar un dataset donde:
- la **feature** (`X`) sea el string (nombre o apellido), y
- el **target** (`y`) sea la etiqueta `"nombre"` o `"apellido"`.

Para eso, tomamos las columnas `nombre` y `apellido` por separado, les asignamos su etiqueta correspondiente, y las apilamos en un único DataFrame. 🧱


In [ ]:
df_nombres = df[["nombre"]].copy()
df_nombres["target"] = "nombre"
df_nombres.rename(columns={"nombre": "feature"}, inplace=True)

df_apellidos = df[["apellido"]].copy()
df_apellidos["target"] = "apellido"
df_apellidos.rename(columns={"apellido": "feature"}, inplace=True)

df_total = pd.concat([df_nombres, df_apellidos], axis=0).dropna()
df_total.shape


In [ ]:
features = df_total["feature"].values
labels = df_total["target"].values
len(features), len(labels)


## ✂️ Separación en train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.2, random_state=10, stratify=labels
)


## 🤖 Modelo 1: ¿Nombre o Apellido?

Armamos un `Pipeline` que combina:

1. 🔢 **`CountVectorizer`**: convierte cada string en un vector de conteos de n-gramas. Acá usamos `ngram_range=(2, 4)` con `analyzer='char'` (n-gramas de **carácter**, no de palabra).
2. 🎲 **`MultinomialNB`**: el clasificador Naive Bayes que vimos arriba.

⚠️ **Por qué char y no word:** cada valor de `feature` es una sola palabra (un nombre o un apellido suelto). Si usáramos n-gramas de *palabra*, el único "1-grama" posible sería la palabra completa (`"Perez"` → `["perez"]`), lo cual no le da al modelo ninguna señal reutilizable entre distintos nombres. Con n-gramas de *carácter* en cambio, el modelo puede aprender que terminaciones como `ez`, `as`, `os` son típicas de apellidos, y eso sí generaliza entre miles de apellidos distintos que comparten esas terminaciones.

## 📝 N-gramas aplicados a nombres

### ¿Qué es un n-grama?

Un **n-grama** es una secuencia contigua de *n* elementos extraídos de un texto. Estos elementos pueden ser:

- **Palabras** (n-gramas a nivel de palabra)
- **Caracteres** (n-gramas a nivel de carácter)

La variable *n* indica cuántos elementos agrupamos:

| n | Nombre técnico |
|---|-----------------|
| 1 | Unigrama |
| 2 | Bigrama |
| 3 | Trigrama |
| n | n-grama |

### ¿Por qué usar n-gramas con nombres?

Cuando trabajamos con datasets de nombres (personas, entidades, empresas), los n-gramas nos permiten:

1. **Detectar similitud entre nombres** aunque tengan errores de tipeo, abreviaciones o distinto orden (ej: "Juan Pérez" vs "Pérez, Juan").
2. **Hacer matching difuso (fuzzy matching)** para deduplicación o record linkage.
3. **Capturar patrones morfológicos** del idioma (prefijos, sufijos, raíces comunes en apellidos).
4. **Alimentar modelos de clasificación** (por ejemplo, predecir si un token es nombre o apellido, o el género, la nacionalidad, etc.) — que es justo lo que hacemos en este notebook. 👇

### N-gramas a nivel de palabra

Se dividen las palabras del texto y se agrupan de a *n*.

**Ejemplo:** `"Juan Carlos Pérez"`

| n | N-gramas resultantes |
|---|------------------------|
| 1 | `Juan`, `Carlos`, `Pérez` |
| 2 | `Juan Carlos`, `Carlos Pérez` |
| 3 | `Juan Carlos Pérez` |

📌 Útil cuando comparamos **strings con varias palabras** y nos interesa el orden entre ellas (ej: comparar "Juan Pérez" vs "Pérez Juan", o detectar coincidencia de nombres compuestos). **No sirve** cuando cada valor ya es una sola palabra suelta (como nuestras columnas `nombre` y `apellido`), porque ahí el único n-grama posible es la palabra entera.

### N-gramas a nivel de carácter

Se recorre el string carácter por carácter, generando ventanas deslizantes de tamaño *n*.

**Ejemplo:** `"Juan"`

| n | N-gramas resultantes |
|---|------------------------|
| 1 | `J`, `u`, `a`, `n` |
| 2 | `Ju`, `ua`, `an` |
| 3 | `Jua`, `uan` |

📌 Este enfoque es el más usado en **record linkage**, **deduplicación de entidades** y **clasificación de tokens cortos**, porque:
- Es robusto a errores de tipeo (`Jaun` vs `Juan` comparten varios trigramas).
- No depende de que las palabras estén bien separadas o en el mismo orden.
- Funciona bien con nombres cortos o abreviados.
- Capta **morfología**: prefijos y sufijos que se repiten dentro de una clase (apellidos que terminan en `-ez`, nombres femeninos que terminan en `-a`, etc.).

### 🎯 Caso específico de este notebook: ¿nombre o apellido? / ¿qué género?

Acá el problema **no es comparar dos strings entre sí**, sino **clasificar un token suelto**: dado `"Perez"`, predecir si es nombre o apellido; dado `"Maria"`, predecir el género más probable. Para esto:

- **N-gramas de palabra no sirven**: cada `feature` ya es una sola palabra, así que el "vector" sería casi siempre un único valor (la palabra completa), sin ninguna generalización posible entre ejemplos distintos.
- **N-gramas de carácter sí sirven**, porque nombres y apellidos en español tienen patrones morfológicos distintos (sobre todo en las terminaciones):
  - Apellidos: terminan mucho en `-ez` (Pérez, González, Martínez), `-as` (Rojas, Salas), `-os` (Ramos, Santos).
  - Nombres: terminan mucho en vocal (`-a`, `-o`, `-e`) o patrones distintos (`-el`, `-án`).
  - Género: terminaciones en `-a` predominan en nombres femeninos, `-o`/consonante en masculinos (con excepciones, por eso es un problema probabilístico y no una regla fija).

**Rango recomendado: `ngram_range=(2, 4)` con `analyzer='char'`**

| Rango (char) | ¿Sirve para clasificar nombre/apellido/género? |
|---|---|
| 1 | ❌ Muy poco — solo cuenta letras sueltas, casi no distingue |
| 1-2 | ⚠️ Débil — bigramas solos pierden terminaciones completas |
| **2-4** | ✅ **Sweet spot** — captura sufijos/prefijos completos (`ez`, `nez`, `Gon`, `Mar`) sin explotar la dimensionalidad |
| 2-5+ | Funciona, pero en nombres cortos empieza a ser redundante con el string completo |

Es exactamente lo que vamos a usar en el `CountVectorizer` de acá abajo. 👇

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

clf = Pipeline([
    ("count_vectorizer", CountVectorizer(ngram_range=(2, 4), analyzer="char")),
    ("bayes", MultinomialNB())
])

clf.fit(X_train, y_train)


In [ ]:
y_pred = clf.predict(X_test)
print(f"🎯 Accuracy: {accuracy_score(y_test, y_pred):.4f}")


### 💾 Guardado y carga del modelo

In [ ]:
import pickle

# Guardamos el modelo entrenado
with open("model_NLP_bayes.pkl", "wb") as f:
    pickle.dump(clf, f)

# Lo volvemos a cargar (simulando un uso posterior, en otro script/servicio)
with open("model_NLP_bayes.pkl", "rb") as f:
    model = pickle.load(f)


In [ ]:
model.predict(["jorge"])[0]


## ⚔️ Comparando modelos: Naive Bayes vs. SGD

¿`MultinomialNB` es realmente la mejor opción? Comparémoslo contra un `SGDClassifier` (un clasificador lineal), probando además `CountVectorizer` vs. `TfidfVectorizer` como vectorizadores. Usamos `cross_val_score` para una comparación más robusta que un único train/test split. 📊


In [ ]:
from sklearn.linear_model import SGDClassifier

sgd = Pipeline([
    ("count_vectorizer", CountVectorizer(ngram_range=(2, 4), analyzer="char")),
    ("sgd", SGDClassifier(loss="modified_huber"))
])
sgd_tfidf = Pipeline([
    ("tfidf_vectorizer", TfidfVectorizer(ngram_range=(2, 4), analyzer="char")),
    ("sgd", SGDClassifier(loss="modified_huber"))
])
NB = Pipeline([
    ("count_vectorizer", CountVectorizer(ngram_range=(2, 4), analyzer="char")),
    ("bayes", MultinomialNB())
])
NB_tfidf = Pipeline([
    ("tfidf_vectorizer", TfidfVectorizer(ngram_range=(2, 4), analyzer="char")),
    ("bayes", MultinomialNB())
])

all_models = [
    ("sgd", sgd),
    ("sgd_tfidf", sgd_tfidf),
    ("NB", NB),
    ("NB_tfidf", NB_tfidf),
]

scores = [(name, cross_val_score(model, X_train, y_train, cv=3).mean()) for name, model in all_models]
sorted(scores, key=lambda x: x[1], reverse=True)


In [ ]:
sgd_tfidf.fit(X_train, y_train)
y_pred = sgd_tfidf.predict(X_test)
print(f"🎯 Accuracy (SGD + TF-IDF): {accuracy_score(y_test, y_pred):.4f}")


In [ ]:
sgd_tfidf.predict(["jorge"])[0]


## 🚻 Modelo 2: Predicción de género a partir del nombre

Ahora entrenamos un segundo modelo, también con `MultinomialNB` y el mismo vectorizador de n-gramas de **carácter** (`ngram_range=(2, 4)`), pero esta vez para predecir el **género (`sexo_id`)** a partir del **nombre**. La misma lógica aplica: ciertas terminaciones y combinaciones de letras (por ejemplo `-a` final en nombres femeninos) son mucho más frecuentes en un género que en otro, y esa es justo la señal estadística que los n-gramas de carácter capturan bien. 📈


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["nombre"], df["sexo_id"], test_size=0.2, random_state=10
)

NB.fit(X_train, y_train)
y_pred = NB.predict(X_test)
print(f"🎯 Accuracy: {accuracy_score(y_test, y_pred):.4f}")


In [ ]:
NB.predict(["MARIA JOSE"])[0]


## 🧪 Vamos a probar el modelo

Esta celda combina ambos modelos: primero determina si el string ingresado es un nombre o un apellido, y si es un nombre, además predice el género más probable. 🕹️


In [ ]:
a = input("✍️  Ingresá un nombre o apellido: ")

if model.predict([a])[0] == "nombre":
    genero = NB.predict([a])[0]
    print(f"✅ '{a}' parece ser un NOMBRE, con mayor probabilidad de pertenecer a: {genero}")
else:
    print(f"✅ '{a}' parece ser un APELLIDO")
